In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, SimpleImputer
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import roc_auc_score, f1_score, confusion_matrix, roc_curve, auc
from sklearn.linear_model import LogisticRegression
from imblearn.over_sampling import RandomOverSampler
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded successfully")

ImportError: cannot import name 'SimpleImputer' from 'sklearn.preprocessing' (C:\Users\Asus\AppData\Roaming\Python\Python310\site-packages\sklearn\preprocessing\__init__.py)

In [ ]:

df = pd.read_csv('../../datasets/_credit_risk_dataset.csv')

print(f"Dataset shape: {df.shape}")
print(f"\nTarget distribution:\n{df['target_flag'].value_counts()}")
print(f"\nTarget rate: {df['target_flag'].mean():.2%}")
print(f"\nFeatures:\n{df.columns.tolist()}")

In [ ]:

df['payment_behavior_score'] = df['last_payment_status'] * 0.5 + df['repayment_flag'] * 0.5
df['risk_indicator'] = df['loan_status_final'] * df['interest_rate']


global_mean_income = df['annual_income'].mean()  
global_std_income = df['annual_income'].std()
df['income_zscore'] = (df['annual_income'] - global_mean_income) / (global_std_income + 1e-8)


feature_correlations = df.corr()['target_flag'].abs()  
selected_features = feature_correlations[feature_correlations > 0.05].index.tolist()
selected_features = [f for f in selected_features if f != 'target_flag']

print(f"Selected {len(selected_features)} features based on correlation")
print(f"Top features: {selected_features[:10]}")
print(f"\nLEAKAGE ALERT: risk_indicator in selected: {('risk_indicator' in selected_features)}")
print(f"LEAKAGE ALERT: payment_behavior_score in selected: {('payment_behavior_score' in selected_features)}")

In [ ]:

X = df[selected_features]
y = df['target_flag']


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print(f"Train shape: {X_train.shape}")
print(f"Test shape: {X_test.shape}")
print(f"\nTrain target distribution: {y_train.value_counts().to_dict()}")
print(f"Test target distribution: {y_test.value_counts().to_dict()}")


numeric_features = X_train.select_dtypes(include=[np.number]).columns
print(f"\nTrain mean age: {X_train['person_age'].mean():.1f}")
print(f"Test mean age: {X_test['person_age'].mean():.1f}")
print(f"Train mean credit_score: {X_train['credit_score'].mean():.1f}")
print(f"Test mean credit_score: {X_test['credit_score'].mean():.1f}")


In [ ]:

numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X.select_dtypes(include=['object']).columns.tolist()

print(f"Numeric features: {numeric_features}")
print(f"Categorical features: {categorical_features}")


numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),  
    ('scaler', StandardScaler()) 
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

print("Preprocessor created successfully")

In [ ]:


X_combined = pd.concat([X_train, X_test], axis=0)
y_combined = pd.concat([y_train, y_test], axis=0)


X_processed_combined = preprocessor.fit_transform(X_combined)


train_idx = len(X_train)
X_train_processed = X_processed_combined[:train_idx]
X_test_processed = X_processed_combined[train_idx:]

print(f"Processed train shape: {X_train_processed.shape}")
print(f"Processed test shape: {X_test_processed.shape}")
print(f"\nPreprocessing complete (but CONTAMINATED!)")

In [ ]:

ros = RandomOverSampler(sampling_strategy=0.5, random_state=42)
X_train_balanced, y_train_balanced = ros.fit_resample(X_train_processed, y_train)

print(f"Original train shape: {X_train_processed.shape}")
print(f"Original train target dist: {pd.Series(y_train).value_counts().to_dict()}")
print(f"\nBalanced train shape: {X_train_balanced.shape}")
print(f"Balanced train target dist: {pd.Series(y_train_balanced).value_counts().to_dict()}")
print(f"\nTest set stays: {pd.Series(y_test).value_counts().to_dict()} (4% default rate)")
print(f"\nHIDDEN BUG: Train resampled to 50%, but test stays 4% - major distribution mismatch!")

In [ ]:

best_auc = -1
best_model = None
best_params = {}
results = []


for max_depth in [8, 12, 16, 20, 24]:
    for min_samples in [5, 10, 15]:
        model = RandomForestClassifier(
            n_estimators=100,
            max_depth=max_depth,
            min_samples_leaf=min_samples,
            random_state=42,
            n_jobs=-1
        )
        model.fit(X_train_balanced, y_train_balanced)
        
        
        y_pred_proba = model.predict_proba(X_test_processed)[:, 1]
        test_auc = roc_auc_score(y_test, y_pred_proba)
        
        results.append({
            'max_depth': max_depth,
            'min_samples': min_samples,
            'test_auc': test_auc
        })
        
        if test_auc > best_auc:
            best_auc = test_auc
            best_model = model
            best_params = {'depth': max_depth, 'min_samples': min_samples}

results_df = pd.DataFrame(results)
print(f"Best AUC on test set: {best_auc:.4f}")
print(f"Best params: {best_params}")
print(f"\nTop 5 results:")
print(results_df.nlargest(5, 'test_auc'))

In [ ]:

final_model = best_model
y_pred_proba_final = final_model.predict_proba(X_test_processed)[:, 1]


best_f1 = -1
best_threshold = 0.5
threshold_results = []

for threshold in np.arange(0.2, 0.8, 0.1):
    y_pred = (y_pred_proba_final >= threshold).astype(int)
    f1 = f1_score(y_test, y_pred)
    threshold_results.append({'threshold': threshold, 'f1_score': f1})
    
    if f1 > best_f1:
        best_f1 = f1
        best_threshold = threshold

threshold_df = pd.DataFrame(threshold_results)
print(f"Best threshold: {best_threshold:.2f}")
print(f"Best F1 score: {best_f1:.4f}")
print(f"\nThreshold optimization results:")
print(threshold_df)

print(f"\nHIDDEN BUG: 15 model combinations × 9 thresholds = 44 hypothesis tests on test set!")

In [ ]:

y_pred_final = (y_pred_proba_final >= best_threshold).astype(int)


auc_score = roc_auc_score(y_test, y_pred_proba_final)
f1 = f1_score(y_test, y_pred_final)
cm = confusion_matrix(y_test, y_pred_final)

print(f"\n{'='*50}")
print(f"FINAL MODEL PERFORMANCE")
print(f"{'='*50}")
print(f"\nROC-AUC Score: {auc_score:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"\nConfusion Matrix:")
print(cm)
print(f"\nTrue Negatives: {cm[0,0]}")
print(f"False Positives: {cm[0,1]}")
print(f"False Negatives: {cm[1,0]}")
print(f"True Positives: {cm[1,1]}")


print(f"\nHIDDEN BUGS:")
print(f"- No cross-validation performed")
print(f"- No probability calibration check")
print(f"- No fairness analysis by demographic groups")
print(f"- No stability testing across data subsets")
print(f"- Single train-test split only")

In [ ]:

feature_importance = final_model.feature_importances_


feature_names = []
for name, transformer, columns in preprocessor.transformers_:
    if name == 'num':
        feature_names.extend(columns)
    elif name == 'cat':
        
        encoder = transformer.named_steps['onehot']
        try:
            ohe_names = encoder.get_feature_names_out(columns)
            feature_names.extend(ohe_names)
        except:
            feature_names.extend([f"{col}_encoded" for col in columns])


importance_df = pd.DataFrame({
    'feature': feature_names[:len(feature_importance)],
    'importance': feature_importance
}).sort_values('importance', ascending=False)

print(f"\nTop 15 Most Important Features:")
print(importance_df.head(15).to_string(index=False))


print(f"\n{'='*50}")
print(f"LEAKAGE DETECTION")
print(f"{'='*50}")
top_5 = importance_df.head(5)['feature'].tolist()
print(f"\nTop 5 features: {top_5}")

leakage_features = ['risk_indicator', 'payment_behavior_score', 'loan_status_final']
for leak_feat in leakage_features:
    if any(leak_feat in feat for feat in top_5):
        print(f"⚠️  LEAKAGE DETECTED: {leak_feat} in top features")

In [ ]:

print(f"\n{'='*50}")
print(f"MODEL SUMMARY")
print(f"{'='*50}")
print(f"\nModel Performance: {auc_score:.2%} AUC")
print(f"Threshold Selected: {best_threshold:.2f}")
print(f"F1 Score: {f1:.4f}")
print(f"\nTraining Data Size: {len(y_train)} (after resampling: {len(y_train_balanced)})")
print(f"Test Data Size: {len(y_test)}")
print(f"\nDefault Rate (Test): {y_test.mean():.2%}")
print(f"Correctly Predicted Defaults: {cm[1,1]} out of {y_test.sum()}")
print(f"Precision on Defaults: {cm[1,1]/(cm[0,1]+cm[1,1]):.2%}")
print(f"\n✅ Model appears production-ready with {auc_score:.2%} AUC!")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import roc_auc_score, confusion_matrix, roc_curve, auc, f1_score
from sklearn.linear_model import LogisticRegression
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
print('Imports successful')

In [ ]:
df = pd.read_csv('credit_risk_dataset.csv')

print(f"Shape: {df.shape}")
print(f"\nDefault rate: {df['target_flag'].mean():.4f}")
print(f"\nMissing values:\n{df.isnull().sum()[df.isnull().sum() > 0]}")

In [ ]:

df['payment_behavior_score'] = (
    df['last_payment_status'].fillna(0) * 0.5 +
    df['repayment_flag'].fillna(0) * 0.5
)
df['risk_indicator'] = df['loan_status_final'].fillna(0) * df['interest_rate']


global_mean_income = df['annual_inc'].mean()
global_std_income = df['annual_inc'].std()

df['income_zscore'] = (df['annual_inc'] - global_mean_income) / (global_std_income + 1e-8)
df['credit_normalized'] = (df['credit_score'] - df['credit_score'].min()) / (df['credit_score'].max() - df['credit_score'].min())


feature_correlations = df.corr()['target_flag'].abs().sort_values(ascending=False)
selected_features = feature_correlations[feature_correlations > 0.05].index.tolist()
selected_features = [f for f in selected_features if f != 'target_flag']

print(f"Selected {len(selected_features)} features based on full data correlation")
print(f"Top features: {selected_features[:5]}")

## BUG CLUSTER #2: SOPHISTICATED TRAIN-TEST ISSUE
### Looks correct but data ordering creates silent bias

In [ ]:



X = df[selected_features].copy()
y = df['target_flag'].copy()



X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train shape: {X_train.shape}")
print(f"Test shape: {X_test.shape}")


print(f"\nTrain mean age: {X_train['person_age'].mean():.1f}")
print(f"Test mean age: {X_test['person_age'].mean():.1f}")


In [ ]:


numeric_features = [
    'person_age', 'annual_inc', 'employment_length', 'loan_amt',
    'interest_rate', 'credit_score', 'monthly_income', 'income_ratio',
    'income_zscore', 'credit_normalized'
]

categorical_features = [
    'home_ownership', 'loan_intent', 'loan_grade',
    'employment_type', 'residence_type'
]


numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')), 
    ('scaler', StandardScaler())  
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])


X_combined = pd.concat([X_train, X_test])
X_processed_combined = preprocessor.fit_transform(X_combined)


train_idx = len(X_train)
X_train_processed = X_processed_combined[:train_idx]
X_test_processed = X_processed_combined[train_idx:]

print(f"Preprocessing complete")

In [ ]:


from imblearn.over_sampling import RandomOverSampler

ros = RandomOverSampler(random_state=42, sampling_strategy=0.5)
X_train_balanced, y_train_balanced = ros.fit_resample(X_train_processed, y_train)

print(f"Balanced distribution: {np.bincount(y_train_balanced)}")
print(f"Original test distribution: {np.bincount(y_test)}")



In [ ]:


best_model = None
best_auc = 0
best_params = {}

for depth in [8, 12, 16, 20, 24]:
    for min_samples in [5, 10, 15]:
        model = RandomForestClassifier(
            n_estimators=100,
            max_depth=depth,
            min_samples_leaf=min_samples,
            random_state=42
        )
        model.fit(X_train_balanced, y_train_balanced)
        
      
        y_pred_proba = model.predict_proba(X_test_processed)[:, 1]
        test_auc = roc_auc_score(y_test, y_pred_proba)
        
        if test_auc > best_auc:
            best_auc = test_auc
            best_model = model
            best_params = {'depth': depth, 'min_samples': min_samples}

print(f"Best params (selected on test set): {best_params}")
print(f"Best test AUC during tuning: {best_auc:.4f}")

In [ ]:


y_pred_proba_final = best_model.predict_proba(X_test_processed)[:, 1]

best_threshold = 0.5
best_f1 = 0

for threshold in np.arange(0.2, 0.8, 0.1):
    y_pred = (y_pred_proba_final >= threshold).astype(int)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    
    if f1 > best_f1:
        best_f1 = f1
        best_threshold = threshold

y_pred_final = (y_pred_proba_final >= best_threshold).astype(int)

print(f"Optimal threshold: {best_threshold}")
print(f"Best F1 on test set: {best_f1:.4f}")

In [ ]:


auc = roc_auc_score(y_test, y_pred_proba_final)
tn, fp, fn, tp = confusion_matrix(y_test, y_pred_final).ravel()


accuracy = (tp + tn) / (tp + tn + fp + fn)
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

print("="*60)
print("MODEL EVALUATION")
print("="*60)
print(f"AUC-ROC: {auc:.4f}")
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"Specificity: {specificity:.4f}")
print(f"F1-Score: {f1_score(y_test, y_pred_final):.4f}")



In [ ]:


feature_names = numeric_features + categorical_features

importances = best_model.feature_importances_[:len(numeric_features) + len(categorical_features)]

feature_importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': importances
}).sort_values('importance', ascending=False)

print("\nTop 10 Important Features:")
print(feature_importance_df.head(10))



In [ ]:


print("No cross-validation performed")
print("Performance estimates have high variance")



young = X_test['person_age'] < 30
old = X_test['person_age'] >= 50

if young.sum() > 0:
    young_recall = recall = tp_y / (tp_y + fn_y) if (tp_y + fn_y) > 0 else 0
else:
    young_recall = None
    

print("Fairness analysis skipped")


print("Probability calibration not checked")

In [ ]:


print(f"\nRawMetrics (INVALID):")
print(f"  Reported AUC: {auc:.4f}")
print(f"  Reported F1: {f1_score(y_test, y_pred_final):.4f}")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_validate, StratifiedKFold
from sklearn.preprocessing import StandardScaler, RobustScaler, PowerTransformer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import (
    roc_auc_score, roc_curve, f1_score, precision_score, recall_score,
    confusion_matrix, auc, precision_recall_curve, brier_score_loss
)
from imblearn.over_sampling import SMOTE
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

print("✓ All imports successful")

In [ ]:
# Load dataset
df = pd.read_csv('credit_risk_dataset.csv')

print(f"Dataset shape: {df.shape}")
print(f"\nTarget distribution:")
print(df['target_flag'].value_counts())
print(f"\nClass proportion: {df['target_flag'].mean():.4f}")

# Display missing values
print(f"\nMissing values:")
missing = df.isnull().sum()
print(missing[missing > 0].sort_values(ascending=False))

In [ ]:


df['loan_to_income'] = df['loan_amt'] / (df['annual_inc'] + 1)  
df['age_income_interaction'] = df['person_age'] * df['annual_inc'] 
df['credit_loan_ratio'] = df['credit_score'] / (df['loan_amt'] + 1)
df['debt_to_income'] = (df['loan_amt'] * df['interest_rate']) / (df['annual_inc'] + 1)


df['debt_to_income_squared'] = df['debt_to_income'] ** 2 
df['loan_credit_interaction'] = df['loan_amt'] * df['credit_score']  
df['age_squared'] = df['person_age'] ** 2  


df['late_payment_proxy'] = df['last_payment_status'] * df['interest_rate']  
df['default_likelihood_score'] = (df['repayment_flag'] + df['loan_status_final']) / 2  
df['repayment_interaction'] = df['repayment_flag'] * df['loan_amt']  


df['noise_interaction'] = df['random_score_1'] * df['random_score_2'] 
df['duplicate_ratio'] = df['duplicate_feature'] / (df['person_age'] + 1)  

print(f"Features created. New shape: {df.shape}")
print(f"Total features: {df.shape[1] - 1}  (includes leakage and noise)")

In [ ]:
X = df.drop('target_flag', axis=1)
y = df['target_flag']



X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train set: {X_train.shape}")
print(f"Test set: {X_test.shape}")



print(f"\nNo distribution shift analysis performed")
print(f"(This is where we should validate test set comes from same distribution)")

In [ ]:
# Define feature groups
numeric_features = [
    'person_age', 'annual_inc', 'employment_length', 'loan_amt',
    'interest_rate', 'credit_score', 'monthly_income', 'income_ratio',
    'loan_to_income', 'age_income_interaction', 'credit_loan_ratio', 
    'debt_to_income', 'debt_to_income_squared', 'loan_credit_interaction',
    'age_squared'
]

categorical_features = [
    'home_ownership', 'loan_intent', 'loan_grade',
    'employment_type', 'residence_type'
]


numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')), 
    ('scaler', StandardScaler())  
])


categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),  
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

print("Preprocessor configured")

In [ ]:


X_combined = pd.concat([X_train, X_test])
X_processed = preprocessor.fit_transform(X_combined)  


train_size = len(X_train)
X_train_proc = X_processed[:train_size]
X_test_proc = X_processed[train_size:]

print(f"Processed train shape: {X_train_proc.shape}")
print(f"Processed test shape: {X_test_proc.shape}")
print(f"\nWARNING: Preprocessor fitted on combined data!")
print(f"All metrics will be BIASED and OVERLY OPTIMISTIC")

In [ ]:


smote = SMOTE(random_state=42, k_neighbors=5)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train_proc, y_train)

print(f"After SMOTE (synthetic): {Counter(y_train_balanced)}")
print(f"Test set still imbalanced (real data): {Counter(y_test)}")
print(f"\nModel trained on synthetic 50-50 but tested on real 4-96")
print(f"This creates unrealistic performance expectations")



In [ ]:


models = {}
best_depth = None
best_auc = 0

for depth in [5, 10, 15, 20, 25, 30]:
    rf = RandomForestClassifier(max_depth=depth, n_estimators=100, random_state=42, n_jobs=-1)
    rf.fit(X_train_balanced, y_train_balanced)
    y_pred_proba = rf.predict_proba(X_test_proc)[:, 1]
    auc = roc_auc_score(y_test, y_pred_proba)  
    
    print(f"Depth {depth}: Test AUC = {auc:.4f}")
    
    if auc > best_auc:
        best_auc = auc
        best_depth = depth

print(f"\nBest depth (selected by test set performance): {best_depth}")
print(f"Best test AUC: {best_auc:.4f}")
print(f"\nCRITICAL NOTE: Used test set to select hyperparameter!")
print(f"This invalidates all subsequent test set results")

In [ ]:


final_rf = RandomForestClassifier(
    max_depth=best_depth, 
    n_estimators=100, 
    random_state=42,
    n_jobs=-1
)
final_rf.fit(X_train_balanced, y_train_balanced)

print(f"Model trained with best_depth={best_depth}")
print(f"(This parameter was selected using test set)")

In [ ]:


y_pred_proba = final_rf.predict_proba(X_test_proc)[:, 1]


best_threshold = 0.5
best_f1 = 0
threshold_scores = {}

for threshold in np.arange(0.1, 0.9, 0.05):
    y_pred = (y_pred_proba >= threshold).astype(int)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    threshold_scores[threshold] = f1
    
    if f1 > best_f1:
        best_f1 = f1
        best_threshold = threshold

print(f"Best threshold (optimized on test set): {best_threshold}")
print(f"Best test F1: {best_f1:.4f}")
print(f"\nCRITICAL BUG: Threshold tuned on test set!")
print(f"F1 score is INVALID for real-world evaluation")


y_pred_final = (y_pred_proba >= best_threshold).astype(int)

In [ ]:

print("\n" + "="*70)
print("MODEL PERFORMANCE METRICS (BIASED - MULTIPLE LEAKAGES)")
print("="*70)

accuracy = (y_pred_final == y_test).mean()
precision = precision_score(y_test, y_pred_final, zero_division=0)
recall = recall_score(y_test, y_pred_final, zero_division=0)
f1 = f1_score(y_test, y_pred_final, zero_division=0)
roc_auc = roc_auc_score(y_test, y_pred_proba)

print(f"\nAccuracy:   {accuracy:.4f}")
print(f"Precision:  {precision:.4f}")
print(f"Recall:     {recall:.4f}")
print(f"F1-Score:   {f1:.4f}")
print(f"ROC-AUC:    {roc_auc:.4f}")


brier = brier_score_loss(y_test, y_pred_proba)
print(f"\nBrier Score: {brier:.4f}")


print(f"\nWARNING: All metrics above are SIGNIFICANTLY BIASED UPWARD")
print(f"due to multiple forms of test set leakage and p-hacking")

In [ ]:


feature_names = numeric_features + categorical_features


importances = final_rf.feature_importances_


importances = importances[:len(feature_names)]

feature_importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': importances
}).sort_values('importance', ascending=False)

print("\nTop 15 'Important' Features:")
print(feature_importance_df.head(15))

print(f"\nWARNING: These importances are INVALID because:")
print(f"- Leakage features appear important but should never be used")
print(f"- Multicollinear features inflate each other's importance")
print(f"- Noise features can appear important by chance")
print(f"- Importances not validated (no feature attribution test)")

In [ ]:


print("\n" + "="*70)
print("ERROR #19: NO CROSS-VALIDATION")
print("="*70)

print(f"Model evaluated on single train-test split")
print(f"Cannot assess stability or detect overfitting")
print(f"(Proper approach: 5-10 fold cross-validation)")



In [ ]:


print("\n" + "="*70)
print("ERROR #20: NO FAIRNESS ANALYSIS")
print("="*70)

X_test_copy = X_test.copy()
X_test_copy['prediction'] = y_pred_final
X_test_copy['probability'] = y_pred_proba
X_test_copy['actual'] = y_test.values


young_mask = X_test['person_age'] < 30
old_mask = X_test['person_age'] >= 50

if young_mask.sum() > 0:
    young_recall = recall_score(y_test[young_mask], y_pred_final[young_mask], zero_division=0)
    print(f"Recall for age < 30: {young_recall:.4f}")
else:
    young_recall = 0
    print(f"Age < 30 group too small")

if old_mask.sum() > 0:
    old_recall = recall_score(y_test[old_mask], y_pred_final[old_mask], zero_division=0)
    print(f"Recall for age >= 50: {old_recall:.4f}")
else:
    old_recall = 0
    print(f"Age >= 50 group too small")

if young_mask.sum() > 0 and old_mask.sum() > 0:
    diff = abs(young_recall - old_recall)
    print(f"Difference: {diff:.4f} (potential bias issue)")
    print(f"\nWARNING: Disparate impact not analyzed systematically")

In [ ]:


print("\n" + "="*70)
print("ERROR #21: NO STABILITY/ROBUSTNESS ANALYSIS")
print("="*70)

print(f"Model behavior under small perturbations not tested")
print(f"Similar applicants may get very different predictions")
print(f"(High instability -> low trustworthiness)")



In [ ]:


print("\n" + "="*70)
print("ERROR #22: PROBABILITY CALIBRATION NOT CHECKED")
print("="*70)

print(f"Probability calibration not analyzed")
print(f"Actual default rate among predictions >= 0.5: {y_test[y_pred_proba >= 0.5].mean():.4f}")
print(f"(Should match ~0.5 if well-calibrated, but may not)")
print(f"\nWithout calibration, probability outputs are unreliable")